# Stage 1 — Awareness: Agent Walkthrough

**Book**: *Mastering Agentic AI for Customer Journey Marketing*  
**Author**: Pushparajan Ramar  
**Chapters**: 3–4  
**Framework**: OpenAI Agents SDK  

---

This notebook walks through every component of the Awareness stage
pipeline, from intent scoring to triage routing.

## 1. Environment Setup

In [ ]:
import os
import sys
import json

# Ensure mock mode is enabled for the walkthrough
os.environ["USE_MOCK_APIS"] = "true"

# Add the project root to the path so imports work
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"USE_MOCK_APIS: {os.getenv('USE_MOCK_APIS')}")

## 2. Intent Scoring

The intent scoring engine aggregates behavioural signals from multiple
sources and produces a weighted 0–100 score.

**Weight model:**
| Signal | Weight |
|--------|--------|
| pricing_page | 25 |
| demo_page | 20 |
| content_download | 15 |
| g2_research | 10 |
| email_open | 5 |

In [ ]:
from stage1_awareness.tools.intent_scoring import score_intent_signals

# Score three leads at different intent levels
leads = ["alice@techcorp.com", "bob@startup.io", "charlie@bigco.org"]

for email in leads:
    profile = score_intent_signals(email)
    print(f"\n{'='*50}")
    print(f"Email: {profile['email']}")
    print(f"Score: {profile['score']}")
    print(f"Tier:  {profile['tier']}")
    print(f"Signals: {json.dumps(profile['signals'], indent=2)}")

## 3. Company Enrichment

The web search tools provide firmographic and technographic data
to help the Discovery Agent match leads against the ICP.

In [ ]:
from stage1_awareness.tools.web_search_tools import search_company_info, get_tech_stack

domain = "techcorp.com"
company = search_company_info(domain)
tech = get_tech_stack(domain)

print("Company Info:")
print(json.dumps(company, indent=2))
print("\nTech Stack:")
print(json.dumps(tech, indent=2))

## 4. CRM Operations

Create, read, and update contacts in HubSpot (mock mode).

In [ ]:
from stage1_awareness.tools.crm_tools import (
    create_crm_contact,
    get_contact,
    update_contact_stage,
)

# Create a contact
crm_result = create_crm_contact(
    email="alice@techcorp.com",
    firstname="Alice",
    lastname="Chen",
    company="TechCorp Inc.",
    source="google_ads",
)
print("Created:", json.dumps(crm_result, indent=2))

# Look up the contact
found = get_contact("alice@techcorp.com")
print("\nLookup:", json.dumps(found, indent=2))

# Update stage
updated = update_contact_stage(crm_result["contact_id"], "lead")
print("\nUpdated:", json.dumps(updated, indent=2))

## 5. Ad Platform Tools

Retrieve click context from Google / Meta Ads and create
retargeting audiences.

In [ ]:
from stage1_awareness.tools.ad_platform_tools import (
    get_ad_click_context,
    create_retargeting_audience,
)

# Google Ads click
click = get_ad_click_context("gclid_abc123")
print("Google Ads click:")
print(json.dumps(click, indent=2))

# Meta Ads click
click2 = get_ad_click_context("fbclid_xyz789")
print("\nMeta Ads click:")
print(json.dumps(click2, indent=2))

# Create retargeting audience
audience = create_retargeting_audience(
    segment_name="High_Intent_Visitors_Q2",
    contacts=["alice@techcorp.com", "bob@startup.io"],
)
print("\nRetargeting audience:")
print(json.dumps(audience, indent=2))

## 6. Brand Safety Guardrail

All outbound content is validated against brand safety rules
before being sent to prospects.

In [ ]:
from stage1_awareness.guardrails.brand_safety import brand_safety_check

# Clean content
good = brand_safety_check(
    "Hi Alice, I noticed TechCorp has been evaluating marketing automation "
    "solutions. Our platform helps enterprise teams streamline demand gen."
)
print("Clean content:", json.dumps(good, indent=2))

# Content with issues
bad = brand_safety_check(
    "ACT NOW! GUARANTEED RESULTS! We are 10x better than CompetitorA! "
    "This is a LIMITED TIME ONLY offer!!!"
)
print("\nBad content:", json.dumps(bad, indent=2))

## 7. Full Triage Pipeline (Direct / Mock Mode)

Run the complete awareness pipeline for all three sample leads
using direct function calls (no LLM required).

In [ ]:
from stage1_awareness.project_demand_gen.main import run_direct_triage, SAMPLE_LEADS

results = []
for lead in SAMPLE_LEADS:
    result = run_direct_triage(lead)
    results.append(result)
    print(f"\n{'='*60}")
    print(f"Lead: {result['email']}")
    print(f"Score: {result['intent']['score']} ({result['intent']['tier']})")
    print(f"Action: {result['action']}")

print(f"\n{'='*60}")
print("Pipeline complete: processed", len(results), "leads")

## 8. Agent-Based Triage (Requires OpenAI API Key)

When `USE_MOCK_APIS=false` and a valid `OPENAI_API_KEY` is set,
the pipeline uses the OpenAI Agents SDK to orchestrate the
Discovery, Triage, Accelerator, and Nurture agents.

Uncomment and run the cell below to test with real LLM calls.

In [ ]:
# # Uncomment to run with real agents (requires OPENAI_API_KEY)
# import asyncio
# os.environ["USE_MOCK_APIS"] = "false"
# os.environ["OPENAI_API_KEY"] = "sk-..."  # Replace with your key
#
# from stage1_awareness.agents.triage_agent import run_triage
#
# lead = {
#     "email": "alice@techcorp.com",
#     "firstname": "Alice",
#     "lastname": "Chen",
#     "company": "TechCorp Inc.",
#     "industry": "Technology",
#     "title": "VP Marketing",
#     "source": "google_ads",
# }
# result = asyncio.get_event_loop().run_until_complete(run_triage(lead))
# print(json.dumps(result, indent=2, default=str))

---

## Summary

In this walkthrough we demonstrated:

1. **Intent Scoring** — Weighted multi-signal scoring (pricing page, demo page, content downloads, G2 intent, email opens).
2. **Company Enrichment** — Firmographic and technographic look-ups.
3. **CRM Operations** — Contact creation, lookup, and stage updates in HubSpot.
4. **Ad Platform Tools** — Click context retrieval and retargeting audience creation.
5. **Brand Safety** — Rule-based content validation guardrail.
6. **Full Pipeline** — End-to-end triage routing with three intent tiers.

Next: **Stage 2 — Consideration** continues the journey with deeper engagement agents.